# Executive Overview

This notebook analyzes the overall performance of the insurance recovery process.

Business Questions:

- How many claims exist?
- How much money is recoverable?
- How much has been collected?
- What is the collection efficiency?
- What is the monthly recovery trend?


## Imports and loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
df=pd.read_excel("../data/claims.xlsx")

## Calculating KPIs

In [2]:
total_claims = df["Claim ID"].nunique()

total_recovery = df["Recovery Amount"].sum()

total_collected = df["Collected Amount"].sum()

total_remaining = df["Remaining Amount"].sum()

collection_rate = (total_collected / total_recovery) * 100

outstanding_rate = (total_remaining / total_recovery) * 100

average_recovery = df["Recovery Amount"].mean()

closed_cases = (
    df["Status"]
    .fillna("")
    .str.strip()
    .str.lower()
    .eq("closed")
    .sum()
)

pending_cases = total_claims - closed_cases

## Display KPIs

In [3]:
kpis = pd.DataFrame({
    "Metric":[
        "Total Claims",
        "Total Recovery",
        "Collected",
        "Remaining",
        "Collection %",
        "Outstanding %",
        "Average Recovery",
        "Closed Cases",
        "Pending Cases"
    ],
    "Value":[
        total_claims,
        total_recovery,
        total_collected,
        total_remaining,
        round(collection_rate,2),
        round(outstanding_rate,2),
        average_recovery,
        closed_cases,
        pending_cases
    ]
})

kpis

,Metric,Value
0,Total Claims,3.267000e+03
1,Total Recovery,3.573387e+07
2,Collected,1.486426e+06
3,Remaining,3.424543e+07
4,Collection %,4.160000e+00
5,Outstanding %,9.583000e+01
6,Average Recovery,1.093782e+04
7,Closed Cases,0.000000e+00
8,Pending Cases,3.267000e+03


### Q1) How much money has actually been recovered?

In [4]:
financial_summary = pd.DataFrame({
    "Category": [
        "Collected",
        "Remaining"
    ],
    "Amount": [
        total_collected,
        total_remaining
    ]
})

fig = px.bar(
    financial_summary,
    x="Category",
    y="Amount",
    text="Amount",
    title="Collected vs Remaining Recovery Amount"
)

fig.update_traces(texttemplate='%{text:,.0f}')

fig.show()

### Q2) How are claims distributed by status?

In [5]:
status_counts = (
    df["Status"]
    .fillna("Unknown")
    .value_counts()
    .reset_index()
)

status_counts.columns = ["Status", "Count"]

fig = px.pie(
    status_counts,
    names="Status",
    values="Count",
    hole=0.55,
    title="Claim Status Distribution"
)

fig.show()

In [6]:
monthly_recovery = (
    df 
    .assign(AccidentDate=pd.to_datetime(df["Accident Date"], errors="coerce"))
    .groupby(pd.Grouper(key="AccidentDate", freq="M"))["Recovery Amount"]
    .sum()
    .reset_index()
)

monthly_recovery["Accident Date"] = monthly_recovery["AccidentDate"].dt.to_period("M").astype(str)

monthly_recovery["Accident Date"] = (
    monthly_recovery["Accident Date"]
    .astype(str)
)

fig = px.line(
    monthly_recovery,
    x="Accident Date",
    y="Recovery Amount",
    markers=True,
    title="Monthly Recovery Trend"
)

fig.show()

C:\Users\Subhan\AppData\Local\Temp\ipykernel_40876\1726662143.py:4: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [7]:
print(f"""
EXECUTIVE INSIGHTS

• Total Claims: {total_claims:,}

• Collection Rate:
{collection_rate:.2f}%

• Outstanding Balance:
SAR {total_remaining:,.0f}

• Outstanding Rate:
{outstanding_rate:.2f}%

• Average Recovery per Claim:
SAR {average_recovery:,.0f}
""")


EXECUTIVE INSIGHTS

• Total Claims: 3,267

• Collection Rate:
4.16%

• Outstanding Balance:
SAR 34,245,431

• Outstanding Rate:
95.83%

• Average Recovery per Claim:
SAR 10,938



In [8]:
def format_currency(value):
    if value >= 1_000_000:
        return f"{value/1_000_000:.2f} M SAR"
    elif value >= 1000:
        return f"{value/1000:.2f} K SAR"
    return f"{value:.2f} SAR"

In [9]:
format_currency(total_recovery)

'35.73 M SAR'